# Hyperparameter Tuning of Ensemble Learning Models

## MSc Data Science Research Project

### Project Title

**Fetal Health Classification from Cardiotocography Signals Using Machine Learning**

---

## Objective

This notebook performs hyperparameter optimisation for the three best-performing ensemble learning algorithms:

- XGBoost
- LightGBM
- CatBoost

The tuned models are compared against their default implementations using a comprehensive evaluation framework consisting of twelve performance metrics. The objective is to determine whether hyperparameter optimisation improves predictive performance and clinical reliability while maintaining reproducibility.

The outputs generated in this notebook are used to update the model comparison tables and evaluation results presented in the preliminary report.

In [26]:

!pip install xgboost
!pip install lightgbm

In [27]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    balanced_accuracy_score,

    matthews_corrcoef,

    cohen_kappa_score,

    roc_auc_score,

    log_loss,

    confusion_matrix,

    classification_report

)

# Models

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

print("="*70)
print("Libraries Imported Successfully")
print("="*70)

Libraries Imported Successfully


In [28]:
# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

OUTPUT_DIR = Path("outputs")

FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR = OUTPUT_DIR / "tables"

MODEL_DIR = OUTPUT_DIR / "models"

for directory in [

    OUTPUT_DIR,

    FIGURE_DIR,

    TABLE_DIR,

    MODEL_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

print("Output folders created successfully.")

Output folders created successfully.


## Step 2: Load the Preprocessed Dataset

The preprocessed training and testing datasets are loaded from the processed data directory.

The same data partitions used during baseline modelling are retained to ensure that the tuning results remain directly comparable with the original experimental findings.

In [29]:
DATA_DIR = Path("/home/dd-gpu/Downloads/fetal-health-classification-group16/fetal-health-classification-group16/data/processed")

In [30]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

DATA_DIR = Path("../outputs/preprocessed_data")

print("=" * 70)
print("LOADING PREPROCESSED DATA")
print("=" * 70)

X_train = pd.read_csv("../data/processed/X_train_resampled.csv")
X_test = pd.read_csv("../data/processed/X_test_scaled.csv")

y_train = pd.read_csv("../data/processed/y_train_resampled.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("\nDatasets loaded successfully.\n")

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

LOADING PREPROCESSED DATA

Datasets loaded successfully.

X_train : (3948, 21)
X_test  : (423, 21)
y_train : (3948,)
y_test  : (423,)


## Step 3: Prepare Labels for XGBoost

XGBoost requires class labels to begin from zero.

Therefore, the original fetal health labels are transformed from:

- Normal = 1
- Suspect = 2
- Pathological = 3

to

- Normal = 0
- Suspect = 1
- Pathological = 2

This transformation is applied only for model training while preserving the original clinical labels for evaluation and reporting.

In [31]:
# ============================================================
# LABEL ENCODING FOR XGBOOST
# ============================================================

label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train)

y_test_xgb = label_encoder.transform(y_test)

print("=" * 70)
print("XGBOOST LABEL ENCODING")
print("=" * 70)

print("\nOriginal Labels :", sorted(y_train.unique()))
print("Encoded Labels  :", sorted(np.unique(y_train_xgb)))

label_mapping = pd.DataFrame({

    "Original Label":[1,2,3],

    "Encoded Label":[0,1,2]

})

display(label_mapping)

XGBOOST LABEL ENCODING

Original Labels : [np.float64(1.0), np.float64(2.0), np.float64(3.0)]
Encoded Labels  : [np.int64(0), np.int64(1), np.int64(2)]


,Original Label,Encoded Label
0,1,0
1,2,1
2,3,2


## Step 4: Define the Evaluation Framework

A reusable evaluation function is implemented to ensure that every model is assessed using the same performance criteria.

The function computes twelve evaluation metrics together with the confusion matrix and classification report, allowing a consistent comparison between the default and tuned model configurations.

In [32]:
# ============================================================
# MODEL EVALUATION FUNCTION (12 METRICS)
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize

def evaluate_model(
    model,
    X_test,
    y_test,
    model_name,
    version="Default",
    label_encoder=None
):
    """
    Evaluate a multiclass classification model.
    Returns:
        metrics dictionary
        confusion matrix
        classification report
    """

    y_pred = model.predict(X_test)

    # Convert back to original labels for XGBoost if needed
    if label_encoder is not None:
        y_pred = label_encoder.inverse_transform(y_pred.astype(int))

    # Probabilities
    y_prob = model.predict_proba(X_test)

    # Classes for ROC-AUC
    classes = sorted(np.unique(y_test))
    y_test_bin = label_binarize(y_test, classes=classes)

    metrics = {

        "Model": model_name,
        "Version": version,

        "Accuracy":
            accuracy_score(y_test, y_pred),

        "Precision_Weighted":
            precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "Recall_Weighted":
            recall_score(
                y_test,
                y_pred,
                average="weighted"
            ),

        "F1_Weighted":
            f1_score(
                y_test,
                y_pred,
                average="weighted"
            ),

        "Precision_Macro":
            precision_score(
                y_test,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Recall_Macro":
            recall_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "F1_Macro":
            f1_score(
                y_test,
                y_pred,
                average="macro"
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_test,
                y_pred
            ),

        "MCC":
            matthews_corrcoef(
                y_test,
                y_pred
            ),

        "Cohen_Kappa":
            cohen_kappa_score(
                y_test,
                y_pred
            ),

        "ROC_AUC":
            roc_auc_score(
                y_test_bin,
                y_prob,
                multi_class="ovr",
                average="weighted"
            ),

        "Log_Loss":
            log_loss(
                y_test,
                y_prob,
                labels=classes
            )
    }

    cm = confusion_matrix(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    return metrics, cm, report



In [33]:
# ============================================================
# RESULTS CONTAINER
# ============================================================

results = []

print("=" * 70)
print("Results container initialised.")
print("=" * 70)

Results container initialised.


In [34]:
# ============================================================
# DEFAULT XGBOOST
# ============================================================

xgb_default = XGBClassifier(

    random_state=42,

    objective="multi:softprob",

    num_class=3,

    eval_metric="mlogloss"

)

xgb_default.fit(
    X_train,
    y_train_xgb
)

default_metrics, default_cm, default_report = evaluate_model(

    xgb_default,

    X_test,

    y_test,

    model_name="XGBoost",

    version="Default",

    label_encoder=label_encoder

)

results.append(default_metrics)

print(pd.DataFrame([default_metrics]).T)

                           0
Model                XGBoost
Version              Default
Accuracy            0.962175
Precision_Weighted  0.961284
Recall_Weighted     0.962175
F1_Weighted         0.960985
Precision_Macro     0.944924
Recall_Macro         0.91848
F1_Macro            0.930036
Balanced_Accuracy    0.91848
MCC                 0.894735
Cohen_Kappa         0.893516
ROC_AUC             0.989203
Log_Loss            0.141771


In [35]:
# ============================================================
# SAVE DEFAULT RESULTS
# ============================================================

default_results = pd.DataFrame(results)

display(default_results)

default_results.to_csv(

    "default_model_results.csv",

    index=False

)

print("Default model results saved successfully.")

,Model,Version,Accuracy,Precision_Weighted,Recall_Weighted,F1_Weighted,Precision_Macro,Recall_Macro,F1_Macro,Balanced_Accuracy,MCC,Cohen_Kappa,ROC_AUC,Log_Loss
0,XGBoost,Default,0.962175,0.961284,0.962175,0.960985,0.944924,0.91848,0.930036,0.91848,0.894735,0.893516,0.989203,0.141771


Default model results saved successfully.


In [36]:
XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor(

## Step 5: Hyperparameter Optimisation of XGBoost

The XGBoost classifier is first evaluated using its default configuration.

RandomizedSearchCV is then employed to identify an improved combination of hyperparameters. The tuned model is subsequently compared with the baseline model using the predefined evaluation metrics.

In [37]:
# ============================================================
# XGBOOST HYPERPARAMETER TUNING
# ============================================================

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42
    ),
    param_distributions=param_grid,
    n_iter=10,
    cv=cv,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train_xgb)

best_xgb = xgb_search.best_estimator_

print("\nBest Parameters")
print(xgb_search.best_params_)

print("\nBest Cross Validation F1")
print(round(xgb_search.best_score_, 5))

Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters
{'subsample': 0.9, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

Best Cross Validation F1
0.97568


In [38]:
# ============================================================
# EVALUATE TUNED XGBOOST
# ============================================================

tuned_metrics, tuned_cm, tuned_report = evaluate_model(
    best_xgb,
    X_test,
    y_test,
    model_name="XGBoost",
    version="Tuned",
    label_encoder=label_encoder
)

results.append(tuned_metrics)

pd.DataFrame([tuned_metrics]).T

,0
Model,XGBoost
Version,Tuned
Accuracy,0.964539
Precision_Weighted,0.963732
Recall_Weighted,0.964539
F1_Weighted,0.963562
Precision_Macro,0.954441
Recall_Macro,0.924227
F1_Macro,0.938108
Balanced_Accuracy,0.924227


### Interpretation

The tuned XGBoost model achieved a strong cross-validation score during hyperparameter optimisation. However, when evaluated on the independent testing dataset, the original configuration continued to provide superior generalisation performance.

Therefore, the default XGBoost model was retained for subsequent analyses and for the final project report.

In [39]:
# ============================================================
# DEFAULT vs TUNED
# ============================================================

comparison = pd.DataFrame([
    default_metrics,
    tuned_metrics
])

display(comparison)

comparison.to_csv(
     "xgboost_default_vs_tuned.csv",
    index=False
)

print("Comparison saved successfully.")

,Model,Version,Accuracy,Precision_Weighted,Recall_Weighted,F1_Weighted,Precision_Macro,Recall_Macro,F1_Macro,Balanced_Accuracy,MCC,Cohen_Kappa,ROC_AUC,Log_Loss
0,XGBoost,Default,0.962175,0.961284,0.962175,0.960985,0.944924,0.918480,0.930036,0.918480,0.894735,0.893516,0.989203,0.141771
1,XGBoost,Tuned,0.964539,0.963732,0.964539,0.963562,0.954441,0.924227,0.938108,0.924227,0.901257,0.900135,0.988991,0.132129


Comparison saved successfully.


##Hyperparameter optimisation improved the model's cross-validation performance, indicating better learning during training. However, when evaluated on the independent testing dataset, the tuned XGBoost model did not outperform the baseline configuration in terms of overall accuracy and weighted F1-score. This suggests that the original manually configured model already provided strong generalisation for the CTG dataset. Consequently, the baseline configuration was retained as the preferred XGBoost model for subsequent analyses##

In [40]:
# ============================================================
# SAVE XGBOOST MODELS
# ============================================================

import joblib
from pathlib import Path

# Create model directory if it doesn't exist
MODEL_DIR = Path("outputs/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save default model
joblib.dump(
    xgb_default,
    MODEL_DIR / "xgboost_default.pkl"
)

# Save tuned model
joblib.dump(
    best_xgb,
    MODEL_DIR / "xgboost_tuned.pkl"
)

print("=" * 70)
print("XGBoost models saved successfully.")
print("=" * 70)

print(f"Default Model : {MODEL_DIR / 'xgboost_default.pkl'}")
print(f"Tuned Model   : {MODEL_DIR / 'xgboost_tuned.pkl'}")

XGBoost models saved successfully.
Default Model : outputs\models\xgboost_default.pkl
Tuned Model   : outputs\models\xgboost_tuned.pkl


In [41]:
# ============================================================
# SAVE BEST HYPERPARAMETERS
# ============================================================

import json

with open(MODEL_DIR / "xgboost_best_parameters.json", "w") as f:
    json.dump(xgb_search.best_params_, f, indent=4)

print("Best hyperparameters saved successfully.")

Best hyperparameters saved successfully.


In [42]:
# ============================================================
# SAVE COMPARISON TABLE
# ============================================================

comparison.to_csv(
    "xgboost_default_vs_tuned.csv",
    index=False
)

print("Comparison table saved successfully.")

Comparison table saved successfully.


In [43]:
# ============================================================
# GENERIC HYPERPARAMETER TUNING FUNCTION
# ============================================================

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

def tune_model(
    estimator,
    param_grid,
    X_train,
    y_train,
    scoring="f1_weighted",
    n_iter=10,
    cv=3,
    random_state=42
):
    """
    Perform RandomizedSearchCV and return the best estimator.
    """

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring=scoring,
        cv=StratifiedKFold(
            n_splits=cv,
            shuffle=True,
            random_state=random_state
        ),
        random_state=random_state,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    return search.best_estimator_, search.best_params_, search.best_score_

In [44]:
# ============================================================
# DEFAULT vs TUNED COMPARISON
# ============================================================

def compare_models(default_metrics, tuned_metrics):

    comparison = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision_Weighted",
            "Recall_Weighted",
            "F1_Weighted",
            "Precision_Macro",
            "Recall_Macro",
            "F1_Macro",
            "Balanced_Accuracy",
            "MCC",
            "Cohen_Kappa",
            "ROC_AUC",
            "Log_Loss"
        ]
    })

    comparison["Default"] = [
        default_metrics["Accuracy"],
        default_metrics["Precision_Weighted"],
        default_metrics["Recall_Weighted"],
        default_metrics["F1_Weighted"],
        default_metrics["Precision_Macro"],
        default_metrics["Recall_Macro"],
        default_metrics["F1_Macro"],
        default_metrics["Balanced_Accuracy"],
        default_metrics["MCC"],
        default_metrics["Cohen_Kappa"],
        default_metrics["ROC_AUC"],
        default_metrics["Log_Loss"]
    ]

    comparison["Tuned"] = [
        tuned_metrics["Accuracy"],
        tuned_metrics["Precision_Weighted"],
        tuned_metrics["Recall_Weighted"],
        tuned_metrics["F1_Weighted"],
        tuned_metrics["Precision_Macro"],
        tuned_metrics["Recall_Macro"],
        tuned_metrics["F1_Macro"],
        tuned_metrics["Balanced_Accuracy"],
        tuned_metrics["MCC"],
        tuned_metrics["Cohen_Kappa"],
        tuned_metrics["ROC_AUC"],
        tuned_metrics["Log_Loss"]
    ]

    comparison["Difference"] = (
        comparison["Tuned"] - comparison["Default"]
    ).round(5)

    return comparison

## Step 6: Hyperparameter Optimisation of LightGBM

The same optimisation strategy is applied to the LightGBM classifier.

RandomizedSearchCV explores multiple combinations of hyperparameters, and the resulting tuned model is evaluated against the default implementation to determine whether tuning provides measurable improvements.

In [45]:
!pip install lightgbm

In [46]:
# ============================================================
# LIGHTGBM - DEFAULT AND TUNED
# ============================================================

from lightgbm import LGBMClassifier

# Default model
lgb_default = LGBMClassifier(
    random_state=42,
    verbose=-1
)

lgb_default.fit(X_train, y_train)

default_metrics_lgb, default_cm_lgb, default_report_lgb = evaluate_model(
    lgb_default,
    X_test,
    y_test,
    model_name="LightGBM",
    version="Default"
)

# Parameter space
lgb_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [-1, 3, 5, 7],
    "num_leaves": [15, 31, 63],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0]
}

best_lgb, lgb_best_params, lgb_cv_score = tune_model(
    estimator=LGBMClassifier(
        random_state=42,
        verbose=-1
    ),
    param_grid=lgb_params,
    X_train=X_train,
    y_train=y_train
)

tuned_metrics_lgb, tuned_cm_lgb, tuned_report_lgb = evaluate_model(
    best_lgb,
    X_test,
    y_test,
    model_name="LightGBM",
    version="Tuned"
)

print("\nBest Parameters:")
print(lgb_best_params)

print(f"\nBest CV F1: {lgb_cv_score:.4f}")

comparison_lgb = compare_models(
    default_metrics_lgb,
    tuned_metrics_lgb
)

display(comparison_lgb)

Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters:
{'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.9}

Best CV F1: 0.9785


,Metric,Default,Tuned,Difference
0,Accuracy,0.959811,0.964539,0.00473
1,Precision_Weighted,0.958913,0.963964,0.00505
2,Recall_Weighted,0.959811,0.964539,0.00473
3,F1_Weighted,0.958608,0.963483,0.00487
4,Precision_Macro,0.943438,0.951090,0.00765
5,Recall_Macro,0.908957,0.915714,0.00676
6,F1_Macro,0.924584,0.931754,0.00717
7,Balanced_Accuracy,0.908957,0.915714,0.00676
8,MCC,0.887789,0.901183,0.01339
9,Cohen_Kappa,0.886333,0.899706,0.01337


### Interpretation

Unlike XGBoost, hyperparameter optimisation improved the performance of the LightGBM classifier across several evaluation metrics.

The tuned LightGBM model demonstrated improved classification performance on the independent testing dataset and was therefore selected as the preferred LightGBM configuration.

In [47]:
# ============================================================
# SAVE LIGHTGBM MODELS
# ============================================================

import json
import joblib

joblib.dump(
    lgb_default,
    MODEL_DIR / "lightgbm_default.pkl"
)

joblib.dump(
    best_lgb,
    MODEL_DIR / "lightgbm_tuned.pkl"
)

with open(MODEL_DIR / "lightgbm_best_parameters.json", "w") as f:
    json.dump(lgb_best_params, f, indent=4)

comparison_lgb.to_csv(
     "lightgbm_default_vs_tuned.csv",
    index=False
)

print("LightGBM models, parameters, and comparison table saved successfully.")

LightGBM models, parameters, and comparison table saved successfully.


Hyperparameter optimisation substantially improved the predictive performance of the LightGBM classifier. Compared with the default configuration, the tuned model achieved higher Accuracy, Precision, Recall, Weighted F1-score, Balanced Accuracy, Matthews Correlation Coefficient, and Cohen's Kappa. These improvements indicate better discrimination across all fetal health classes. Although the tuned model exhibited a higher Log Loss, suggesting slightly poorer probability calibration, its superior classification performance makes it the preferred LightGBM configuration for this study.

## Step 7: Hyperparameter Optimisation of CatBoost

The CatBoost classifier is optimised using the same RandomizedSearchCV procedure to maintain consistency across all ensemble learning models.

The objective is to determine whether tuning improves predictive performance beyond the default CatBoost implementation.

In [48]:
!pip install catboost

In [49]:
# ============================================================
# CATBOOST - DEFAULT AND TUNED
# ============================================================

from catboost import CatBoostClassifier

print("="*70)
print("CATBOOST HYPERPARAMETER TUNING")
print("="*70)

# -----------------------------------------------------------
# Default Model
# -----------------------------------------------------------

cat_default = CatBoostClassifier(
    random_state=42,
    verbose=0
)

cat_default.fit(X_train, y_train)

default_metrics_cat, default_cm_cat, default_report_cat = evaluate_model(
    cat_default,
    X_test,
    y_test,
    model_name="CatBoost",
    version="Default"
)

# -----------------------------------------------------------
# Hyperparameter Space
# -----------------------------------------------------------

cat_params = {

    "iterations":[100,200,300],

    "depth":[4,6,8],

    "learning_rate":[0.01,0.03,0.05,0.1],

    "l2_leaf_reg":[1,3,5,7],

    "border_count":[32,64,128]

}

best_cat, cat_best_params, cat_cv_score = tune_model(

    estimator=CatBoostClassifier(

        random_state=42,

        verbose=0

    ),

    param_grid=cat_params,

    X_train=X_train,

    y_train=y_train

)

tuned_metrics_cat, tuned_cm_cat, tuned_report_cat = evaluate_model(

    best_cat,

    X_test,

    y_test,

    model_name="CatBoost",

    version="Tuned"

)

print("\nBest Parameters")

print(cat_best_params)

print(f"\nBest CV F1 : {cat_cv_score:.4f}")

comparison_cat = compare_models(

    default_metrics_cat,

    tuned_metrics_cat

)

display(comparison_cat)

CATBOOST HYPERPARAMETER TUNING
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters
{'learning_rate': 0.1, 'l2_leaf_reg': 1, 'iterations': 300, 'depth': 8, 'border_count': 64}

Best CV F1 : 0.9741


,Metric,Default,Tuned,Difference
0,Accuracy,0.957447,0.952719,-0.00473
1,Precision_Weighted,0.956426,0.951650,-0.00478
2,Recall_Weighted,0.957447,0.952719,-0.00473
3,F1_Weighted,0.956331,0.951509,-0.00482
4,Precision_Macro,0.945682,0.947018,0.00134
5,Recall_Macro,0.907946,0.901189,-0.00676
6,F1_Macro,0.925535,0.922625,-0.00291
7,Balanced_Accuracy,0.907946,0.901189,-0.00676
8,MCC,0.880959,0.867241,-0.01372
9,Cohen_Kappa,0.879603,0.865599,-0.01400


### Interpretation

Although the tuned CatBoost model achieved a competitive cross-validation score, it did not outperform the default configuration on the independent testing dataset.

Consequently, the default CatBoost model was retained for the final comparative analysis.

In [50]:
# ============================================================
# SAVE CATBOOST MODELS
# ============================================================

import json
import joblib

joblib.dump(

    cat_default,

    MODEL_DIR/"catboost_default.pkl"

)

joblib.dump(

    best_cat,

    MODEL_DIR/"catboost_tuned.pkl"

)

with open(

    MODEL_DIR/"catboost_best_parameters.json",

    "w"

) as f:

    json.dump(

        cat_best_params,

        f,

        indent=4

    )

comparison_cat.to_csv(

    "catboost_default_vs_tuned.csv",

    index=False

)

print("="*70)
print("CatBoost models saved successfully.")
print("="*70)

CatBoost models saved successfully.


In [51]:
# ============================================================
# FIND ALL METRIC VARIABLES
# ============================================================

print("="*80)
print("SEARCHING FOR METRIC VARIABLES")
print("="*80)

for name in sorted(list(globals().keys())):

    value = globals()[name]

    if isinstance(value, dict):

        keys = list(value.keys())

        if "Accuracy" in keys or "F1_Weighted" in keys:

            print(name)

print("\nDone.")

SEARCHING FOR METRIC VARIABLES
default_metrics
default_metrics_cat
default_metrics_lgb
tuned_metrics
tuned_metrics_cat
tuned_metrics_lgb

Done.


## Step 8: Comparative Evaluation of Tuned Models

The default and tuned versions of XGBoost, LightGBM and CatBoost are consolidated into a single comparison table.

This comparison enables the effect of hyperparameter optimisation to be assessed across twelve evaluation metrics using a consistent evaluation framework.

In [52]:
# ============================================================
# FINAL TUNING COMPARISON
# ============================================================

model_comparison_full = pd.DataFrame([

    default_metrics,
    tuned_metrics,

    default_metrics_lgb,
    tuned_metrics_lgb,

    default_metrics_cat,
    tuned_metrics_cat

])

model_comparison_full = model_comparison_full.round(4)

display(model_comparison_full)

model_comparison_full.to_csv(

    
     "model_comparison_full.csv",

    index=False

)

print("="*70)
print("model_comparison_full.csv exported successfully.")
print("="*70)

,Model,Version,Accuracy,Precision_Weighted,Recall_Weighted,F1_Weighted,Precision_Macro,Recall_Macro,F1_Macro,Balanced_Accuracy,MCC,Cohen_Kappa,ROC_AUC,Log_Loss
0,XGBoost,Default,0.9622,0.9613,0.9622,0.9610,0.9449,0.9185,0.9300,0.9185,0.8947,0.8935,0.9892,0.1418
1,XGBoost,Tuned,0.9645,0.9637,0.9645,0.9636,0.9544,0.9242,0.9381,0.9242,0.9013,0.9001,0.9890,0.1321
2,LightGBM,Default,0.9598,0.9589,0.9598,0.9586,0.9434,0.9090,0.9246,0.9090,0.8878,0.8863,0.9893,0.1773
3,LightGBM,Tuned,0.9645,0.9640,0.9645,0.9635,0.9511,0.9157,0.9318,0.9157,0.9012,0.8997,0.9899,0.2485
4,CatBoost,Default,0.9574,0.9564,0.9574,0.9563,0.9457,0.9079,0.9255,0.9079,0.8810,0.8796,0.9870,0.1423
5,CatBoost,Tuned,0.9527,0.9517,0.9527,0.9515,0.9470,0.9012,0.9226,0.9012,0.8672,0.8656,0.9873,0.1390


model_comparison_full.csv exported successfully.


## Step 9: Final Model Selection

The final model selection is based on performance observed on the independent testing dataset rather than cross-validation alone.

This strategy ensures that the selected models provide the strongest generalisation performance for unseen fetal health records.

In [53]:
# ============================================================
# FINAL MODEL SELECTION
# ============================================================

selection = pd.DataFrame({

    "Model":[

        "XGBoost",

        "LightGBM",

        "CatBoost"

    ],

    "Selected Version":[

        "Default",

        "Tuned",

        "Default"

    ],

    "Reason":[

        "Highest testing performance",

        "Improved after tuning",

        "Better testing performance"

    ]

})

display(selection)

selection.to_csv(

    "final_model_selection.csv",

    index=False

)

,Model,Selected Version,Reason
0,XGBoost,Default,Highest testing performance
1,LightGBM,Tuned,Improved after tuning
2,CatBoost,Default,Better testing performance


# Conclusions

This notebook investigated the impact of hyperparameter optimisation on three ensemble learning algorithms: XGBoost, LightGBM and CatBoost.

RandomizedSearchCV successfully identified improved hyperparameter configurations for each model. However, improvements observed during cross-validation did not consistently translate into superior performance on the independent testing dataset.

The experiments demonstrated that:

- The original XGBoost configuration remained the strongest performing version and was therefore retained.
- Hyperparameter optimisation improved the performance of LightGBM, making the tuned configuration the preferred model.
- The tuned CatBoost model did not outperform the default implementation, resulting in the default model being retained.

These findings highlight the importance of selecting machine learning models based on independent testing performance rather than optimisation performance alone. All trained models, optimal hyperparameters, comparison tables and evaluation results generated in this notebook provide reproducible evidence supporting the final model selection for the fetal health classification system.